# 01 — Árbol de decisión C4.5

Para clasificar un caso nuevo, primero se necesita aprender una relación entre sus atributos y una respuesta conocida. Por ejemplo, si se dispone de solicitudes ya aceptadas o rechazadas, sus datos permiten buscar características que distingan ambas clases. El entrenamiento utiliza esos casos resueltos para construir un modelo. Después, el modelo recibe los atributos de otra solicitud y predice su clase, sin conocer de antemano la respuesta correcta.

Un árbol de decisión organiza ese aprendizaje como una secuencia de preguntas. La primera pregunta ocupa la raíz del árbol y sus respuestas posibles forman las ramas. Cada rama conduce a otra pregunta, representada por un nodo interno, o a una hoja que produce la predicción. Para construir esa estructura se debe decidir qué atributo preguntar, cómo separar sus valores y cuándo dejar de agregar preguntas. El punto de partida es, por tanto, encontrar una forma de comparar las divisiones que cada pregunta produciría en los casos de entrenamiento.

Este notebook implementa esas operaciones en una clase llamada `ArbolC45`, con pandas para organizar los datos y código propio para aprender el árbol. El criterio de selección y el tratamiento de atributos numéricos siguen la presentación de C4.5 de Liu (2011, capítulo 3, sección 3.2); las reglas concretas de esta implementación se explican junto a su código. Se trata de una variante basada en C4.5, con controles de crecimiento, manejo de valores faltantes y pospoda opcional. Primero se desarrolla cómo evaluar una división; después, cómo buscarla en atributos categóricos y numéricos; finalmente, cómo construir el árbol completo y utilizarlo para predecir. Las celdas definen estas operaciones, mientras que sus experimentos y resultados se presentan en `03_experimentos.ipynb`.


In [6]:
# Herramientas para representar nodos y realizar sumas y logaritmos.
from dataclasses import dataclass, field
from math import expm1, fsum, isfinite, log, log2, sqrt
from numbers import Integral, Real
from statistics import NormalDist

# Tablas y comprobaciones de los tipos de sus columnas.
import pandas as pd
from pandas.api.types import is_bool_dtype, is_complex_dtype, is_numeric_dtype, is_scalar


def es_faltante(valor):
    """Reconoce si un valor individual representa información ausente.

    Entrada: el valor de un atributo de un caso. Devuelve True para None,
    NaN, pd.NA y NaT; False para otros valores. Un texto vacío no se
    interpreta automáticamente como ausencia. No cambia el valor recibido.
    """
    # Se comprueba primero que sea un valor individual, no una lista.
    return is_scalar(valor) and bool(pd.isna(valor))


## 1. Cómo evaluar una división

### Medir la mezcla de clases

Para elegir una pregunta, se observa cómo quedarían distribuidas las clases después de responderla. Si una rama reúne solo casos de la clase A, resulta sencillo asignarles esa clase. Si reúne cantidades similares de A y B, todavía queda incertidumbre sobre la respuesta. Por eso se busca que las divisiones produzcan grupos con menor mezcla de clases, medida que se denomina impureza. En esta implementación se utiliza la entropía para cuantificarla, siguiendo a Liu (2011, pp. 72–73).

Se denomina $D$ al conjunto de casos que llega al nodo analizado. En la raíz incluye todos los casos de entrenamiento; más abajo incluye los que llegan por las decisiones anteriores. Si hay $n$ casos y $n_c$ pertenecen a la clase $c$, su proporción es $p_c=n_c/n$. La entropía combina esas proporciones mediante:

$$H(D)=-\sum_c p_c\log_2 p_c.$$

La suma recorre las clases y se toma $0\log_2 0=0$ para una clase ausente. La entropía vale cero si todos los casos pertenecen a la misma clase. Con dos clases, alcanza un bit cuando ambas tienen la misma proporción; la unidad es el bit porque se utiliza logaritmo en base 2. Esta medida permite comparar la mezcla del nodo original con la que queda después de una división.

### Comparar la mezcla antes y después

Una pregunta divide $D$ en grupos $D_j$, uno por rama. Para evaluar el resultado se calcula la entropía de cada grupo y se obtiene un promedio ponderado por su tamaño. Así, una rama que recibe muchos casos pesa más en la evaluación que una que recibe pocos. La diferencia entre la entropía original y ese promedio se denomina ganancia de información, o *Information Gain*:

$$H_A(D)=\sum_j\frac{|D_j|}{|D|}H(D_j),\qquad G(D,A)=H(D)-H_A(D).$$

Aquí $A$ identifica el atributo utilizado y $|D_j|$ indica cuántos casos recibe la rama $j$. Como ejemplo ilustrativo, supóngase que llegan seis casos: cuatro de A y dos de B. La entropía inicial es aproximadamente 0,918 bits. Si una pregunta deja tres casos de A en una rama y un caso de A junto con dos de B en la otra, la entropía posterior es $(3/6)\,0+(3/6)\,0{,}918\simeq0{,}459$. La ganancia es entonces aproximadamente $0{,}918-0{,}459=0{,}459$ bits: la división redujo la mezcla, aunque una rama todavía contiene ambas clases.

### Corregir la preferencia por muchos grupos

Elegir únicamente por ganancia puede favorecer atributos con muchos valores distintos. Un identificador único, por ejemplo, separaría cada caso en su propia rama y dejaría entropía cero en todas ellas. Ese resultado describe perfectamente los casos vistos, pero no ofrece una regla útil para un identificador nuevo. Para moderar esa preferencia se divide la ganancia por una medida de cómo se reparte el conjunto entre las ramas, llamada *Split Info*. El cociente se denomina *Gain Ratio* y es el criterio utilizado aquí para comparar divisiones (Liu, 2011, pp. 74–75):

$$I(D,A)=-\sum_j\frac{|D_j|}{|D|}\log_2\frac{|D_j|}{|D|},\qquad R(D,A)=\frac{G(D,A)}{I(D,A)}.$$

La entropía anterior se calculaba sobre clases; Split Info se calcula sobre las proporciones de las ramas. En el ejemplo de dos ramas de igual tamaño, Split Info vale 1 y Gain Ratio vale aproximadamente 0,459. La búsqueda compara los candidatos y conserva el de mayor Gain Ratio. Se necesitan al menos dos ramas con casos y se descarta una división con Split Info cero, porque no produce una separación válida. Las funciones de cálculo devuelven cero cuando el denominador es cero para evitar una división indefinida.

### Conservar el aporte de los casos con datos faltantes

Hasta aquí se supuso que cada caso podía asignarse a una rama. Si falta el valor del atributo consultado, esa asignación no se conoce, aunque se conozcan la clase real y los demás atributos del caso. La implementación conserva esa información repartiendo el aporte del caso entre las ramas. Por ejemplo, si el 60 % de los casos conocidos sigue una rama y el 40 % la otra, un caso con ese atributo ausente aporta un peso de 0,6 a la primera y de 0,4 a la segunda. Ambos aportes mantienen su clase real y suman el peso original, que al comenzar el entrenamiento es 1.

A partir de ese reparto, contar filas ya no describe cuánto aporta cada caso al nodo. Por eso los cálculos anteriores se generalizan mediante pesos. La suma de esos pesos se denomina masa, y sustituye a la cantidad de casos en las proporciones. Se usa la siguiente notación:

| Símbolo | Significado |
|---|---|
| $i$ | Posición de un caso dentro del nodo; no es el índice de pandas. |
| $y_i$ | Clase real del caso, recibida en la columna `objetivo`. |
| $w_i$ | Peso con el que participa el caso en ese nodo. |
| $W=\sum_{i\in D}w_i$ | Masa total del nodo. |
| $W_c=\sum_{i\in D:\,y_i=c}w_i$ | Masa de la clase $c$: se suman solo los pesos de los casos de esa clase. |
| $p_c=W_c/W$ | Proporción de la clase $c$ que se utiliza en la entropía. |

Con pesos unitarios, estas expresiones coinciden con contar casos. Por ejemplo, tres casos de A y dos de B dan $W=5$ y $p_A=3/5$. Con pesos fraccionarios, dos casos de A con pesos 1 y 0,5 y un caso de B con peso 1 dan $W=2{,}5$ y $p_A=1{,}5/2{,}5=0{,}6$. Las proporciones de las clases siguen sumando 1, aunque la masa pueda diferir de la cantidad de filas representadas. El mismo reemplazo se aplica al promedio de entropías y a Split Info, usando $W_j/W$ para el peso relativo de una rama.

### Evaluar un atributo que contiene ausencias

Antes de repartir un caso, se necesita evaluar la división usando los valores que sí se conocen. Para el atributo $A$, se separan los casos conocidos $D_K$ de los desconocidos $D_U$, con masas $W_K$ y $W_U$, de modo que $W=W_K+W_U$. La ganancia se calcula entre los conocidos y luego se multiplica por $W_K/W$. Así se reduce el respaldo asignado a un atributo cuando solo se conoce una parte de sus valores:

$$G^*(D,A)=\frac{W_K}{W}\left[H(D_K)-\sum_j\frac{W_j}{W_K}H(D_j)\right].$$

Para calcular Split Info, los desconocidos se cuentan como un grupo adicional, además de las ramas conocidas. Ese grupo participa en el cálculo, pero no se convierte en una rama del árbol. La medida y el cociente ajustados quedan definidos por:

$$I^*(D,A)=-\sum_j\frac{W_j}{W}\log_2\frac{W_j}{W}-\frac{W_U}{W}\log_2\frac{W_U}{W},\qquad R^*(D,A)=\frac{G^*(D,A)}{I^*(D,A)}.$$

Las proporciones para repartir los casos desconocidos se calculan solo entre conocidos: $q_j=W_j/W_K$. Si finalmente se elige esa división, cada caso desconocido de peso $w_i$ aporta $w_iq_j$ a la rama $j$. Si todos los valores del atributo están ausentes, no hay información con la que proponer una división. Cuando no hay ausencias, estas fórmulas recuperan los cálculos anteriores porque $W_K=W$.

Las funciones de la siguiente celda implementan estos cálculos. `evaluar_particion` recibe clases, pesos y posiciones agrupadas por rama; devuelve ganancia ajustada, Split Info, Gain Ratio y los datos necesarios para repartir los casos. Las clases y los pesos se relacionan por posición, y los buscadores deben entregar grupos disjuntos que cubran los casos de peso positivo, dejando los faltantes aparte. La función valida los pesos, pero presupone esas condiciones sobre las posiciones; la construcción de los grupos se explica a continuación.


In [7]:
def preparar_pesos(etiquetas, pesos=None):
    """Prepara etiquetas y pesos alineados por su posición en el nodo.

    Entradas: etiquetas contiene las clases; pesos contiene un aporte por caso.
    Si pesos es None, asigna 1.0 a cada caso. Devuelve dos listas del mismo
    largo. Un peso puede ser fraccionario o cero, pero no negativo.
    Lanza ValueError ante longitudes distintas, booleanos, pesos no reales,
    infinitos o NaN. La alineación es por posición, no por índice de pandas.
    """
    # Cada caso vale uno, salvo que llegue con un aporte fraccionario.
    etiquetas = list(etiquetas)
    pesos = [1.0] * len(etiquetas) if pesos is None else list(pesos)
    if len(etiquetas) != len(pesos):
        raise ValueError("Cada etiqueta debe tener su peso correspondiente.")
    if any(not isinstance(peso, Real) or isinstance(peso, bool)
           or not isfinite(peso) or peso < 0 for peso in pesos):
        raise ValueError("Los pesos deben ser reales finitos y no negativos.")
    return etiquetas, pesos


def conteos_ponderados(etiquetas, pesos):
    """Suma el aporte positivo de cada clase y devuelve un diccionario.

    Recibe etiquetas y pesos ya preparados y alineados. Cada clave es una
    clase y su valor es la masa total de esa clase. Los pesos cero no aportan
    ni crean una clase en el resultado. Las copias fraccionarias suman su
    aporte real: no se cuentan como casos completos adicionales.
    La función supone que los pesos ya se validaron; no los comprueba otra vez.
    """
    # Agrupar antes de sumar reduce los errores de redondeo.
    aportes_por_clase = {}
    for clase, peso in zip(etiquetas, pesos):
        if peso > 0:
            aportes_por_clase.setdefault(clase, []).append(peso)
    return {clase: fsum(aportes) for clase, aportes in aportes_por_clase.items()}


def entropia_masas(masas):
    """Calcula la entropía de masas no negativas y finitas ya validadas.

    Recibe los aportes de clases o grupos y devuelve la incertidumbre en bits.
    Normaliza cada masa por el total. Los términos cero no contribuyen y una
    lista vacía o de masa total cero devuelve 0.0. Esta función interna supone
    masas válidas; no rechaza por sí misma todas las entradas incorrectas.
    """
    # Sin masa no hay mezcla que medir.
    masas = list(masas)
    masa_total = fsum(masas)
    if masa_total == 0:
        return 0.0
    return -fsum((masa / masa_total) * log2(masa / masa_total)
                 for masa in masas if masa > 0)


def entropia(etiquetas, pesos=None):
    """Calcula H(D), en bits, usando la masa de cada clase.

    Recibe las etiquetas del nodo y, opcionalmente, sus pesos por posición.
    Devuelve un número: cero si no hay masa o si una sola clase aporta masa.
    Las proporciones se calculan con pesos, incluso después de varios repartos
    por faltantes. La validación de pesos se delega a preparar_pesos.
    """
    # La entropía depende de proporciones de clases, no del número de copias.
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    return entropia_masas(conteos_ponderados(etiquetas, pesos).values())


def entropia_ponderada(grupos, pesos_grupos=None):
    """Promedia la entropía de las ramas según sus masas.

    Entradas: grupos contiene una lista de etiquetas por rama; pesos_grupos
    contiene los pesos correspondientes, o None para usar pesos unitarios.
    Devuelve la suma de la entropía de cada rama multiplicada por su proporción
    de masa. Si la masa total es cero, devuelve 0.0. Lanza ValueError si faltan
    listas de pesos, no coinciden sus largos o sus valores son inválidos.
    Al evaluar un atributo con nulos, los grupos recibidos contienen conocidos.
    """
    # Preparar una lista de pesos por rama y comprobar su correspondencia.
    grupos = [list(grupo) for grupo in grupos]
    pesos_grupos = ([[1.0] * len(grupo) for grupo in grupos]
                    if pesos_grupos is None else list(pesos_grupos))
    if len(grupos) != len(pesos_grupos):
        raise ValueError("Cada grupo debe tener sus pesos correspondientes.")
    grupos_preparados = [preparar_pesos(grupo, pesos)
                         for grupo, pesos in zip(grupos, pesos_grupos)]
    masa_total = fsum(fsum(pesos) for _, pesos in grupos_preparados)
    if masa_total == 0:
        return 0.0

    # Una rama con mayor masa tiene más influencia en el promedio.
    return fsum(fsum(pesos) / masa_total * entropia(grupo, pesos)
                for grupo, pesos in grupos_preparados)


def ganancia_informacion(etiquetas, grupos, pesos=None, pesos_grupos=None):
    """Resta la entropía posterior a la entropía inicial, en bits.

    Recibe etiquetas del conjunto, grupos de etiquetas por rama y sus pesos
    opcionales. Devuelve la reducción de incertidumbre. El llamador debe
    asegurar que los grupos cubran el mismo conjunto con los mismos aportes;
    esta función no comprueba esa cobertura. El ajuste por datos desconocidos
    se realiza en evaluar_particion, no en esta resta.
    """
    # Ambas entropías usan los mismos casos y aportes.
    return entropia(etiquetas, pesos) - entropia_ponderada(grupos, pesos_grupos)


def informacion_particion(grupos, pesos_grupos=None):
    """Calcula Split Info: la entropía de las masas de las ramas.

    Recibe etiquetas agrupadas por rama y pesos opcionales por grupo.
    Devuelve un número en bits; con una sola rama positiva o masa cero vale 0.
    Sin pesos utiliza el tamaño de cada grupo. No incorpora por sí misma
    casos desconocidos: evaluar_particion agrega su masa al cálculo completo.
    La cantidad de grupos y las listas de pesos deben corresponderse.
    """
    # Con pesos unitarios basta contar las filas de cada grupo.
    grupos = [list(grupo) for grupo in grupos]
    if pesos_grupos is None:
        return entropia_masas([len(grupo) for grupo in grupos])
    pesos_grupos = list(pesos_grupos)
    if len(grupos) != len(pesos_grupos):
        raise ValueError("Cada grupo debe tener sus pesos correspondientes.")
    return entropia_masas([fsum(preparar_pesos(grupo, pesos)[1])
                          for grupo, pesos in zip(grupos, pesos_grupos)])


def razon_ganancia(etiquetas, grupos, pesos=None, pesos_grupos=None):
    """Calcula Gain Ratio para una partición completa sin desconocidos.

    Recibe etiquetas, grupos por rama y sus pesos opcionales. Devuelve ganancia
    de información dividida por Split Info, sin unidades. Si el denominador
    es cero devuelve 0.0 como protección; eso no vuelve elegible la división.
    El llamador debe asegurar la cobertura de los grupos. Para atributos con
    faltantes, el árbol utiliza la evaluación ajustada de evaluar_particion.
    """
    # Evitar una división por cero cuando no existe separación útil.
    informacion = informacion_particion(grupos, pesos_grupos)
    if informacion == 0:
        return 0.0
    return ganancia_informacion(etiquetas, grupos, pesos, pesos_grupos) / informacion


def evaluar_particion(etiquetas, indices_por_rama, pesos=None, faltantes=()):
    """Evalúa una división conocida y ajusta sus medidas por los faltantes.

    Entradas:
        etiquetas: clases de todos los casos representados en el nodo.
        indices_por_rama: diccionario rama -> posiciones de casos conocidos.
            Las posiciones empiezan en cero dentro del nodo, no son sus índices
            de pandas. Cada conocido con peso positivo pertenece a una sola rama.
        pesos: aporte por posición; None equivale a uno por caso.
        faltantes: posiciones cuyo valor del atributo evaluado se desconoce.
            Deben quedar fuera de las ramas conocidas.

    El llamador debe aportar posiciones válidas, sin repetir ni superponer
    conocidos y desconocidos, y cubrir los casos con masa positiva. La función
    valida los pesos, pero no comprueba esas condiciones sobre las posiciones.

    Devuelve un diccionario con:
        entropia_ponderada: incertidumbre posterior solo de los conocidos.
        ganancia: mejora entre conocidos multiplicada por masa_conocida / total.
        split_info: entropía de las masas conocidas por rama y de la masa faltante.
        gain_ratio: ganancia ajustada / split_info; cero si el denominador es cero.
        ramas: posiciones conocidas por rama, para construir los descendientes.
        faltantes: lista de posiciones que luego deberán repartirse.
        proporciones: masa conocida de cada rama / masa conocida total.

    Si no hay masa conocida, la ganancia es cero y proporciones queda vacío.
    La masa desconocida participa en Split Info como grupo de cálculo, sin
    crear una rama. Esta función evalúa; no copia casos ni reparte sus pesos.
    Los buscadores descartan los atributos sin dos ramas conocidas positivas.
    """
    # Reunir las etiquetas y masas conocidas de cada rama.
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    grupos = [[etiquetas[posicion] for posicion in posiciones]
              for posiciones in indices_por_rama.values()]
    pesos_grupos = [[pesos[posicion] for posicion in posiciones]
                    for posiciones in indices_por_rama.values()]
    posiciones_conocidas = [posicion for posiciones in indices_por_rama.values()
                            for posicion in posiciones]
    masa_conocida = fsum(pesos[posicion] for posicion in posiciones_conocidas)
    masa_faltante = fsum(pesos[posicion] for posicion in faltantes)
    masa_total = masa_conocida + masa_faltante

    # Medir la mejora sobre conocidos y reducirla según su presencia en el nodo.
    ganancia_ajustada = 0.0
    if masa_total > 0 and masa_conocida > 0:
        ganancia_ajustada = masa_conocida / masa_total * ganancia_informacion(
            [etiquetas[posicion] for posicion in posiciones_conocidas], grupos,
            [pesos[posicion] for posicion in posiciones_conocidas], pesos_grupos)
    informacion = entropia_masas([fsum(pesos_rama) for pesos_rama in pesos_grupos]
                                 + [masa_faltante])

    # Guardar también lo necesario para construir los hijos sin recalcular cortes.
    return {
        "entropia_ponderada": entropia_ponderada(grupos, pesos_grupos),
        "ganancia": ganancia_ajustada,
        "split_info": informacion,
        "gain_ratio": ganancia_ajustada / informacion if informacion > 0 else 0.0,
        "ramas": indices_por_rama,
        "faltantes": list(faltantes),
        "proporciones": {
            rama: fsum(pesos[posicion] for posicion in posiciones) / masa_conocida
            for rama, posiciones in indices_por_rama.items()
        } if masa_conocida else {},
    }


# Nombres habituales conservados como accesos a las mismas funciones.
split_info = informacion_particion
gain_ratio = razon_ganancia


## 2. Cómo se buscan las preguntas candidatas

Las medidas anteriores permiten comparar divisiones, pero todavía se necesita definir cuáles probar. La forma de hacerlo depende del tipo de atributo: una categoría propone ramas por valor, mientras que una medida numérica propone un corte. El tipo se determina a partir de la columna de pandas: los tipos numéricos, salvo booleanos, reciben el tratamiento numérico; las demás columnas reciben el categórico. Por ello, un número que representa un código de categoría debe declararse como texto o `category` si se quiere evitar interpretarlo como una medida.

### Atributos categóricos

Para un atributo categórico se agrupan los casos por su valor conocido. Si un atributo contiene rojo, verde y azul, se propone una rama por cada uno de esos valores. Las clases y los pesos de cada grupo permiten calcular las medidas de la sección anterior; los casos con valor faltante se conservan aparte para el ajuste y el reparto. Una vez elegido el atributo, se retira de ese camino: dentro de cada rama todos sus valores conocidos ya coinciden y volver a preguntar lo mismo no los separaría. Si hay menos de dos categorías conocidas con peso positivo, ese atributo no propone una división.

### Atributos numéricos

Para un atributo numérico se busca un umbral $t$ que forme dos ramas: valores menores o iguales a $t$ y valores mayores que $t$. Aunque una medida admita infinitos valores posibles, en el nodo solo hay una cantidad finita de valores observados. Se ordenan sus valores conocidos distintos $v_1<\cdots<v_r$ y se prueban los $r-1$ cortes $t=v_1,\ldots,v_{r-1}$. Por ejemplo, con valores 10, 20 y 35 se prueban los cortes 10 y 20; usar 35 dejaría vacía la rama derecha. Este tratamiento sigue el procedimiento descrito por Liu (2011, pp. 75–76).

Cada corte se evalúa con la misma ganancia ajustada y el mismo Gain Ratio que una división categórica. Se conserva el mejor corte del atributo y después se compara con los candidatos de los demás atributos. Si el atributo numérico resulta elegido, sigue disponible en sus descendientes: otro corte puede separar un intervalo más pequeño. Un valor exactamente igual al umbral siempre sigue la rama `<=`. La implementación usa valores observados como umbrales, sin calcular puntos medios, redondear decimales ni convertir enteros grandes a flotantes para elegir el corte.

### Orden de selección y filtro opcional

La configuración inicial elige por Gain Ratio tanto el corte de cada atributo numérico como el atributo que dividirá el nodo. Si dos cortes empatan, se conserva el menor, porque se recorren en orden creciente. Si dos atributos empatan, se conserva el primero según el orden de las columnas. Estas reglas hacen que la selección sea determinista para una misma entrada y configuración. La decisión utiliza únicamente la información del nodo actual; elegir el mejor candidato en ese punto no implica haber encontrado el mejor árbol entre todos los posibles.

El parámetro `filtro_ganancia=True` permite estudiar una variante de esa selección. Primero elige el corte numérico de mayor ganancia ajustada, en lugar de elegirlo por Gain Ratio. Después calcula la ganancia media de los mejores candidatos por atributo y conserva los que alcanzan esa media, con tolerancia de $10^{-12}$ para el redondeo. Entre los candidatos que quedan, elige el de mayor Gain Ratio, manteniendo las reglas de empate anteriores. El filtro está desactivado inicialmente y se evalúa como configuración experimental; no garantiza una mejora en casos nuevos.

La celda siguiente define `particion_categorica` y `buscar_mejor_umbral`, que devuelven los campos de evaluación o `None` si no existe una separación válida. Se ignoran los casos de peso cero, los valores repetidos no generan cortes duplicados y un atributo completamente vacío no aporta candidatos. La búsqueda numérica vuelve a recorrer los casos para evaluar cada corte; con $n$ casos y $r$ valores distintos, su costo aproximado por atributo y nodo es $O(n\log n+rn)$. Este costo corresponde al procedimiento implementado, que no reutiliza conteos acumulados entre cortes. Una vez elegida la pregunta, se necesita guardar su resultado en un nodo.


In [8]:
def particion_categorica(valores, etiquetas, pesos=None):
    """Agrupa los valores conocidos y evalúa una división por categoría.

    Recibe valores de un atributo, etiquetas y pesos opcionales alineados por
    posición. Los valores conocidos deben poder usarse como claves de un
    diccionario. Ignora pesos cero y separa las posiciones faltantes.
    Devuelve el diccionario de evaluar_particion, o None si quedan menos de
    dos categorías conocidas con masa positiva, incluso si hay muchos nulos.
    Lanza ValueError si no coincide el número de valores y etiquetas o si
    los pesos son inválidos. El entrenamiento valida antes las categorías.
    """
    # Comprobar que cada valor tenga una etiqueta y un peso.
    valores = list(valores)
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")

    # Guardar los ausentes aparte: luego su peso se repartirá entre los hijos.
    ramas, posiciones_faltantes = {}, []
    for posicion, (valor, peso) in enumerate(zip(valores, pesos)):
        if peso == 0:
            continue
        if es_faltante(valor):
            posiciones_faltantes.append(posicion)
        else:
            ramas.setdefault(valor, []).append(posicion)
    if len(ramas) < 2:
        return None
    return evaluar_particion(etiquetas, ramas, pesos, posiciones_faltantes)


def buscar_mejor_umbral(valores, etiquetas, pesos=None, por_ganancia=False):
    """Busca el corte binario de mayor razón de ganancia para un atributo numérico.

    Entradas: valores contiene las medidas del nodo; etiquetas, sus clases;
    pesos, el aporte de cada caso, o None para usar uno por caso. Las tres
    secuencias se relacionan por posición. Los conocidos con peso positivo
    deben ser números reales finitos y no booleanos; se admiten faltantes.

    Ordena los conocidos distintos y prueba cada valor salvo el mayor como
    umbral t. No redondea los decimales ni calcula puntos medios. Cada candidato
    forma dos ramas no vacías: <= y >. Los valores repetidos permanecen juntos.
    Los faltantes no proponen cortes, pero sí ajustan las medidas de evaluación.

    Devuelve los campos de evaluar_particion más umbral, el valor observado
    elegido. Devuelve None si no hay dos valores conocidos distintos con peso
    positivo: atributo vacío, constante o sin masa suficiente para separar.
    Los pesos cero se ignoran antes de inspeccionar sus valores.

    Lanza ValueError ante longitudes incompatibles, pesos inválidos o conocidos
    numéricos inválidos con peso positivo. Ante Gain Ratio exactamente igual,
    conserva el menor corte porque recorre candidatos en orden creciente.
    Con por_ganancia=True selecciona por ganancia ajustada en lugar del cociente.
    Busca un corte para este nodo; no decide la prepoda ni elimina el atributo,
    que puede volver a dividirse en nodos posteriores.
    """
    # Validar los conocidos y conservar sus posiciones dentro del nodo.
    valores = list(valores)
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")
    posiciones_conocidas, posiciones_faltantes = [], []
    for posicion, (valor, peso) in enumerate(zip(valores, pesos)):
        if peso == 0:
            continue
        if es_faltante(valor):
            posiciones_faltantes.append(posicion)
        elif not isinstance(valor, Real) or isinstance(valor, bool) or not isfinite(valor):
            raise ValueError("Los valores numéricos conocidos deben ser reales y finitos.")
        else:
            posiciones_conocidas.append(posicion)

    # El mayor valor se excluye porque dejaría vacía la rama derecha.
    umbrales_candidatos = sorted({valores[posicion]
                                 for posicion in posiciones_conocidas})[:-1]
    mejor_particion = None
    for umbral in umbrales_candidatos:
        ramas = {
            "<=": [posicion for posicion in posiciones_conocidas if valores[posicion] <= umbral],
            ">": [posicion for posicion in posiciones_conocidas if valores[posicion] > umbral],
        }
        evaluacion = evaluar_particion(etiquetas, ramas, pesos, posiciones_faltantes)
        evaluacion["umbral"] = umbral
        # El criterio alternativo se usa únicamente con el filtro experimental.
        medida_seleccion = "ganancia" if por_ganancia else "gain_ratio"
        if (mejor_particion is None
                or evaluacion[medida_seleccion] > mejor_particion[medida_seleccion]):
            mejor_particion = evaluacion
    return mejor_particion


## 3. Cómo se representa el árbol aprendido

Para conservar las decisiones elegidas, el árbol se construye con objetos `Nodo` conectados mediante sus hijos. Cada nodo guarda un resumen de los casos que llegaron a él: los conteos ponderados por clase, la clase más frecuente, la entropía y el peso total. Ese resumen se calcula incluso si el nodo después se divide, porque puede utilizarse al predecir o al podar. `n_ejemplos` cuenta las filas representadas y `peso_total` suma sus aportes; con valores faltantes, ambas cantidades pueden diferir.

Cuando se elige una división, el nodo guarda además el atributo consultado, su tipo, el umbral si es numérico, las medidas de evaluación y las proporciones de las ramas. Sus hijos representan los grupos resultantes y pueden contener nuevas preguntas. Cuando no se sigue dividiendo, el nodo queda sin hijos y funciona como hoja, con la distribución de clases que ya tenía calculada. El campo `motivo_parada` permite identificar por qué terminó allí el crecimiento o si una pospoda lo convirtió en hoja. Esta estructura permite aplicar el mismo procedimiento de construcción a cada grupo de casos.


In [9]:
@dataclass
class Nodo:
    """Representa una decisión o una hoja con la información aprendida del nodo.

    Atributos de resumen:
        clase: clase mayoritaria del nodo según el orden global de classes_.
        conteos: masa ponderada acumulada por clase en el nodo.
        n_ejemplos: cantidad de filas recibidas (puede diferir de peso_total).
        entropia: incertidumbre de clases en bits calculada con los pesos.
        peso_total: suma de pesos de las filas que llegan al nodo.

    Atributos de decisión (vacíos en hojas):
        atributo, tipo, umbral: pregunta elegida y umbral si es numérico.
        hijos, proporciones: subárboles descendientes y pesos relativos conocidos.
        ganancia, split_info, gain_ratio: métricas ajustadas de la división.
        motivo_parada: razón de detención o 'pospoda' si fue podado a hoja.
    """
    # Resumen de las clases y del aporte de los casos que llegaron hasta aquí.
    clase: object
    conteos: dict
    n_ejemplos: int
    entropia: float
    peso_total: float = 0.0

    # Decisión que conecta este nodo con sus hijos; queda vacía en las hojas.
    atributo: object = None
    tipo: str | None = None
    umbral: float | int | None = None
    hijos: dict = field(default_factory=dict)
    proporciones: dict = field(default_factory=dict)
    ganancia: float | None = None
    split_info: float | None = None
    gain_ratio: float | None = None
    motivo_parada: str | None = None

    @property
    def es_hoja(self):
        """Indica que no existen decisiones descendientes."""
        return not self.hijos


## 4. Cómo se entrena y se utiliza el modelo

### Construcción del árbol y límites de crecimiento

El entrenamiento comienza con todos los casos en la raíz y un peso de 1 para cada uno. Se calcula el resumen de sus clases, se comprueba si corresponde detenerse y, si puede continuar, se busca la mejor división con las funciones anteriores. Cada hijo recibe los casos conocidos de su rama y la fracción de peso correspondiente de los casos con el atributo ausente. A continuación se repite el procedimiento dentro de cada hijo; esta repetición sobre grupos cada vez más específicos es la construcción recursiva del árbol. Si un caso vuelve a repartirse más abajo, se multiplica el peso con el que llegó, conservando su clase y sus otros atributos.

Agregar preguntas permite distinguir más detalles de los casos vistos, pero también puede adaptar el árbol a particularidades que no se repitan en casos nuevos. Para limitar ese crecimiento se pueden fijar condiciones antes de crear los hijos, procedimiento denominado prepoda. Además, la construcción termina cuando todas las clases con peso positivo coinciden, cuando se agotan los atributos disponibles o cuando no existe una división válida. La tabla presenta los controles configurables; las medidas $G^*$ y $R^*$ son las definidas en la sección 1.

| Parámetro | Nombre compatible | Condición y valor inicial |
|---|---|---|
| `profundidad_maxima` | `max_depth` | Detiene al alcanzar la profundidad indicada, contando la raíz como 0. Inicialmente `None`, sin ese límite. |
| `minimo_muestras_division` | `min_samples_split` | Exige una masa mínima de casos para intentar dividir. Es real, finita y de al menos 2; inicialmente 2. |
| `umbral_ganancia` | `threshold` | Detiene si la ganancia ajustada $G^*$ es estrictamente menor que este valor, expresado en bits. Inicialmente 0. |
| `minima_razon_ganancia` | `min_gain_ratio` | Detiene si el Gain Ratio $R^*$ es estrictamente menor que este valor, sin unidades. Inicialmente 0. |
| `filtro_ganancia` | Mismo nombre | Activa el procedimiento alternativo de selección de la sección 2. Inicialmente `False`. |
| `pospoda` | Mismo nombre | Activa la revisión del árbol ya construido que se explica a continuación. Inicialmente `False`. |

La masa mínima se compara con la suma de pesos que recibe el nodo, por lo que admite fracciones. Los límites de ganancia y Gain Ratio controlan cantidades distintas y se validan por separado. Con ambos en cero, se permite una división de ganancia cero, porque una pregunta posterior todavía podría aportar información. Los parámetros se vuelven a validar cada vez que se entrena, y `obtener_parametros` permite consultar la configuración efectiva. El entrenamiento utiliza decisiones locales y límites configurables; su resultado no constituye una búsqueda exhaustiva del mejor árbol posible.

### Revisión del árbol mediante pospoda

También se puede limitar la complejidad después de construir el árbol, revisando si conviene reemplazar un conjunto de decisiones por una hoja. Ese procedimiento se denomina pospoda y se activa con `pospoda=True`. Se recorre el árbol desde las hojas hacia la raíz, para evaluar cada subárbol después de revisar sus descendientes. En cada nodo se compara el error estimado del subárbol con el que tendría una hoja que predijera su clase mayoritaria.

Para estimar ese error, primero se calcula la masa de entrenamiento que la hoja clasificaría mal: el peso total menos el peso de la clase mayoritaria. Se aplica una estimación conservadora basada en una aproximación binomial con corrección de continuidad, siguiendo el criterio documentado para [J48 de Weka](https://weka.sourceforge.io/doc.dev/weka/classifiers/trees/j48/Stats.html). Esta implementación fija el factor de confianza en 0,25 y utiliza la estimación como una regla heurística cuando hay pesos fraccionarios. Si el error estimado de la hoja no supera la suma de los errores estimados de las hojas del subárbol, con tolerancia de $10^{-12}$, se reemplaza ese subárbol por la hoja. La operación conserva la distribución de clases del nodo y utiliza solo resúmenes de entrenamiento; su efecto sobre casos nuevos se debe evaluar por separado.

### Predicción de un caso nuevo

Una vez entrenado el árbol, predecir consiste en seguir sus preguntas usando los atributos del nuevo caso. Un valor conocido indica una rama y permite continuar hasta una hoja. La hoja devuelve la proporción ponderada de cada clase, $P(c)=W_c/W$, y se elige la de mayor proporción. Si hay empate, se utiliza la primera de esas clases según el orden en que apareció en el entrenamiento completo. Estas probabilidades describen las frecuencias aprendidas; no se ha comprobado que estén calibradas para otros casos.

Si falta el atributo consultado, no se puede elegir una única rama. En ese punto se consultan todos los hijos y se combinan sus probabilidades con las proporciones $q_j$ aprendidas al construir el nodo:

$$P(c\mid x,\text{nodo})=\sum_j q_jP(c\mid x,\text{hijo}_j).$$

Aquí $x$ es el caso nuevo y $c$ la clase cuya probabilidad se calcula. Por ejemplo, si dos ramas tienen pesos 0,6 y 0,4 y devuelven probabilidades de A de 0,8 y 0,3, la probabilidad combinada de A es $0{,}6\cdot0{,}8+0{,}4\cdot0{,}3=0{,}6$. Se realiza la misma operación para las demás clases y luego se elige la más probable. Durante esta consulta no se conoce la clase real ni se vuelve a entrenar: se reúnen respuestas del árbol que ya fue aprendido. Si el valor ausente corresponde a otro atributo que el nodo no consulta, se puede seguir su decisión normalmente.

Una categoría nueva requiere otra respuesta, porque tiene un valor presente pero no corresponde a ninguna rama conocida en ese nodo. En ese caso se devuelve la distribución de clases del propio nodo. Para un valor numérico fuera del rango observado, en cambio, los umbrales aprendidos permiten decidir una rama mediante la comparación habitual. Estas reglas permiten responder a consultas distintas de las filas vistas en entrenamiento. Las filas se leen conservando sus tipos individuales y al final se normalizan las probabilidades para absorber errores de redondeo.

### Entradas y salidas de la clase

`entrenar(caracteristicas, objetivos)` recibe dos tablas de pandas (`DataFrame`) con el mismo índice y orden de filas. La primera contiene los atributos y la segunda contiene solo la columna `objetivo`, sin valores faltantes. El método aprende el árbol, reemplaza cualquier árbol anterior del objeto y devuelve el propio modelo. Se admiten ausencias en los atributos, pero se rechazan tablas vacías, nombres de columnas repetidos o nulos, números infinitos y complejos. En las consultas se exige conservar las columnas y su orden, y los atributos aprendidos como numéricos deben seguir conteniendo números.

`predecir` devuelve una tabla con la columna `prediccion`, mientras que `predecir_probabilidades` devuelve una columna por clase, en el orden guardado en `classes_`. Ambos métodos conservan el índice, sus posibles duplicados y el orden de las filas, sin modificar la entrada. Los nombres `fit`, `predict` y `predict_proba` permiten acceder a esas mismas operaciones mediante la terminología habitual de las bibliotecas de aprendizaje automático. Del mismo modo, los nombres compatibles de los parámetros permiten consultar o modificar sus valores. Las siguientes situaciones completan el comportamiento de entrada y consulta:

| Situación | Comportamiento |
|---|---|
| `None`, `NaN`, `pd.NA` o `NaT` en un atributo | Se reconoce como valor faltante y se aplica el tratamiento explicado. |
| Texto vacío o marcador textual como `"?"` | Se considera una categoría; si representa una ausencia, debe convertirse antes a un valor faltante. |
| Valores numéricos repetidos | Permanecen en la misma rama y no generan cortes duplicados. |
| Un único valor conocido distinto, o todos ausentes | El atributo no propone una división; si ningún atributo puede dividir, se crea una hoja. |
| Casos de peso cero | No proponen cortes ni aportan peso a las clases o ramas. |
| Todos los atributos de una consulta ausentes | La combinación de ramas recupera la distribución de clases de la raíz, salvo redondeo. |

La celda siguiente reúne la construcción, la poda y la predicción en `ArbolC45`. Los ejemplos numéricos del texto son ilustrativos y la evaluación experimental se realiza en `03_experimentos.ipynb`. Para la descripción de entropía, Gain Ratio y divisiones numéricas se consultó Liu, B. (2011), *Web Data Mining*, segunda edición, capítulo 3, sección 3.2, disponible en `docs/libros/`. Los controles opcionales, los contratos de entrada y los casos particulares descritos corresponden al código de este proyecto.


In [10]:
class ArbolC45:
    """Árbol propio con categorías, cortes numéricos, nulos y poda opcional.

    Los cuatro parámetros en español controlan el crecimiento. Se admiten
    sus nombres compatibles en inglés, sin permitir configuraciones ambiguas.
    """

    _FACTOR_POSPODA = 0.25
    _CUANTIL_POSPODA = NormalDist().inv_cdf(1 - _FACTOR_POSPODA)

    def __init__(self, umbral_ganancia=0.0, profundidad_maxima=None,
                 minimo_muestras_division=2, minima_razon_ganancia=0.0,
                 filtro_ganancia=False, pospoda=False, **parametros_compatibles):
        """Inicializa el árbol con controles de prepoda, filtro y pospoda opcional.

        Parámetros:
            umbral_ganancia: ganancia ajustada mínima en bits para dividir (>= 0).
            profundidad_maxima: nivel máximo del árbol (None sin límite, >= 0).
            minimo_muestras_division: masa mínima requerida para intentar dividir (>= 2).
            minima_razon_ganancia: Gain Ratio mínimo requerido para dividir (>= 0).
            filtro_ganancia: si es True, filtra candidatas por ganancia media.
            pospoda: si es True, aplica pospoda pesimista tras construir el árbol.
            **parametros_compatibles: alias en inglés (threshold, max_depth,
                min_samples_split, min_gain_ratio), sin duplicar un mismo control.

        Lanza TypeError ante parámetros desconocidos y ValueError ante tipos,
        rangos inválidos o especificaciones ambiguas en español e inglés.
        """
        # Traducir los nombres compatibles sin mantener dos implementaciones.
        parametros = dict(umbral_ganancia=umbral_ganancia,
                          profundidad_maxima=profundidad_maxima,
                          minimo_muestras_division=minimo_muestras_division,
                          minima_razon_ganancia=minima_razon_ganancia)
        equivalencias = {
            "threshold": ("umbral_ganancia", 0.0),
            "max_depth": ("profundidad_maxima", None),
            "min_samples_split": ("minimo_muestras_division", 2),
            "min_gain_ratio": ("minima_razon_ganancia", 0.0),
        }
        for nombre, valor in parametros_compatibles.items():
            if nombre not in equivalencias:
                raise TypeError(f"Parámetro desconocido: {nombre}.")
            nombre_espanol, valor_inicial = equivalencias[nombre]
            if parametros[nombre_espanol] != valor_inicial:
                raise ValueError(f"Usá solo un nombre para {nombre_espanol}.")
            parametros[nombre_espanol] = valor
        for nombre, valor in parametros.items():
            setattr(self, nombre, valor)
        self.filtro_ganancia = filtro_ganancia
        self.pospoda = pospoda
        self.raiz_ = None
        self._validar_parametros()

    # Los nombres compatibles reflejan los mismos parámetros en español.
    @property
    def threshold(self):
        """Acceso compatible al valor de umbral_ganancia."""
        return self.umbral_ganancia

    @threshold.setter
    def threshold(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.umbral_ganancia = valor

    @property
    def max_depth(self):
        """Acceso compatible al valor de profundidad_maxima."""
        return self.profundidad_maxima

    @max_depth.setter
    def max_depth(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.profundidad_maxima = valor

    @property
    def min_samples_split(self):
        """Acceso compatible al valor de minimo_muestras_division."""
        return self.minimo_muestras_division

    @min_samples_split.setter
    def min_samples_split(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.minimo_muestras_division = valor

    @property
    def min_gain_ratio(self):
        """Acceso compatible al valor de minima_razon_ganancia."""
        return self.minima_razon_ganancia

    @min_gain_ratio.setter
    def min_gain_ratio(self, valor):
        """Actualiza el mismo parámetro; entrenar vuelve a validar su valor."""
        self.minima_razon_ganancia = valor

    def obtener_parametros(self):
        """Devuelve los valores efectivos para registrar cómo se construyó el árbol."""
        return {nombre: getattr(self, nombre) for nombre in (
            "umbral_ganancia", "profundidad_maxima",
            "minimo_muestras_division", "minima_razon_ganancia", "filtro_ganancia", "pospoda")}

    def _validar_parametros(self):
        """Rechaza límites negativos, tipos incorrectos y valores no finitos."""
        if not isinstance(self.filtro_ganancia, bool):
            raise ValueError("filtro_ganancia debe ser True o False.")
        if not isinstance(self.pospoda, bool):
            raise ValueError("pospoda debe ser True o False.")
        # Los dos umbrales son números reales; la profundidad es un entero y la masa puede ser fraccionaria.
        for nombre in ("umbral_ganancia", "minima_razon_ganancia"):
            valor = getattr(self, nombre)
            if (not isinstance(valor, Real) or isinstance(valor, bool)
                    or not isfinite(valor) or valor < 0):
                raise ValueError(f"{nombre} debe ser real, finito y no negativo.")
        if self.profundidad_maxima is not None and (
                not isinstance(self.profundidad_maxima, Integral)
                or isinstance(self.profundidad_maxima, bool) or self.profundidad_maxima < 0):
            raise ValueError("profundidad_maxima debe ser None o un entero no negativo.")
        if (not isinstance(self.minimo_muestras_division, Real)
                or isinstance(self.minimo_muestras_division, bool)
                or not isfinite(self.minimo_muestras_division)
                or self.minimo_muestras_division < 2):
            raise ValueError("minimo_muestras_division debe ser real, finito y de al menos 2.")

    @staticmethod
    def _validar_caracteristicas(caracteristicas):
        """Comprueba la tabla de entrada sin modificarla; admite datos ausentes."""
        # Comprobar primero el formato general y los nombres de los atributos.
        if not isinstance(caracteristicas, pd.DataFrame):
            raise TypeError("Las características deben ser un pandas.DataFrame.")
        if caracteristicas.empty:
            raise ValueError("La tabla debe contener casos y atributos.")
        if not caracteristicas.columns.is_unique or caracteristicas.columns.isna().any():
            raise ValueError("Los nombres de los atributos deben ser únicos y no nulos.")

        # Los valores conocidos deben poder compararse o usarse como categorías.
        for columna in caracteristicas.columns:
            serie = caracteristicas[columna]
            if is_complex_dtype(serie.dtype):
                raise ValueError("No se admiten atributos de números complejos.")
            for valor in serie:
                if es_faltante(valor):
                    continue
                if isinstance(valor, complex) or (isinstance(valor, Real) and not isfinite(valor)):
                    raise ValueError("Los números conocidos deben ser reales y finitos.")
                try:
                    hash(valor)
                except TypeError as error:
                    raise ValueError("Las categorías deben ser valores individuales comparables.") from error

    def entrenar(self, caracteristicas, objetivos):
        """Aprende el árbol con dos DataFrames alineados y devuelve este modelo.

        objetivos debe tener una columna objetivo sin nulos. El entrenamiento
        reemplaza cualquier árbol anterior y nunca modifica las tablas recibidas.
        """
        # Validar las entradas antes de establecer el estado del modelo.
        self._validar_parametros()
        self._validar_caracteristicas(caracteristicas)
        if not isinstance(objetivos, pd.DataFrame):
            raise TypeError("Los objetivos deben ser un pandas.DataFrame.")
        if (list(objetivos.columns) != ["objetivo"]
                or not caracteristicas.index.equals(objetivos.index)):
            raise ValueError("objetivos debe tener la columna objetivo y el mismo índice y orden.")
        if objetivos.isna().any().any():
            raise ValueError("Las respuestas correctas no deben contener faltantes.")
        etiquetas = objetivos["objetivo"].tolist()
        try:
            clases = list(dict.fromkeys(etiquetas))
        except TypeError as error:
            raise ValueError("Las clases deben ser valores individuales comparables.") from error

        # Recordar el significado de las columnas y un orden estable de clases.
        self.classes_ = pd.Index(clases, tupleize_cols=False)
        self.columnas_ = caracteristicas.columns.copy()
        self.tipos_ = {
            columna: ("numerico" if is_numeric_dtype(caracteristicas[columna].dtype)
                      and not is_bool_dtype(caracteristicas[columna].dtype) else "categorico")
            for columna in caracteristicas.columns
        }
        self.raiz_ = self._construir(caracteristicas, etiquetas,
                                     list(caracteristicas.columns), [1.0] * len(etiquetas), 0)
        if self.pospoda:
            self._podar(self.raiz_)
        return self

    @classmethod
    def _error_estimado_hoja(cls, nodo):
        """Estima el límite superior de error de un nodo si actuara como hoja.

        Entrada:
            nodo: objeto Nodo con masa total positiva y conteos por clase.

        Calcula la masa mal clasificada por la clase mayoritaria y aplica la
        aproximación binomial con corrección de continuidad de J48/C4.5 usando
        _FACTOR_POSPODA = 0.25 y su cuantil normal _CUANTIL_POSPODA. Cuando el
        error observado es menor que min(1.0, masa), interpola linealmente desde
        la cota para cero errores. Devuelve un float con la masa de error estimada.
        """
        masa = nodo.peso_total
        error = max(0.0, masa - nodo.conteos[nodo.clase])
        # Para error menor que uno se interpola entre cero y el primer caso posible.
        error_cero = -masa * expm1(log(cls._FACTOR_POSPODA) / masa)
        extremo = min(1.0, masa)
        observado = max(error, extremo)
        if observado + 0.5 >= masa:
            estimado = masa
        else:
            proporcion = (observado + 0.5) / masa
            z = cls._CUANTIL_POSPODA
            ajuste = z ** 2 / masa
            limite = (proporcion + ajuste / 2
                      + z * sqrt(proporcion * (1 - proporcion) / masa + (ajuste / 2) ** 2))
            estimado = masa * limite / (1 + ajuste)
        if error < extremo:
            return error_cero + (estimado - error_cero) * (error / extremo)
        return estimado

    def _podar(self, nodo):
        """Poda el subárbol de abajo hacia arriba y devuelve su error estimado.

        Entrada:
            nodo: raíz del subárbol a revisar tras la construcción completa.

        Recorre primero los hijos en post-orden y suma sus errores estimados.
        Si el error estimado de reemplazar el nodo por una hoja no supera al del
        subárbol (con tolerancia 1e-12), limpia hijos, proporciones y métricas,
        marca motivo_parada='pospoda' y devuelve el error de la hoja; en caso
        contrario conserva el subárbol y devuelve la suma de errores de sus ramas.
        """
        error_hoja = self._error_estimado_hoja(nodo)
        if nodo.es_hoja:
            return error_hoja
        error_subarbol = fsum(self._podar(hijo) for hijo in nodo.hijos.values())
        if error_hoja > error_subarbol + 1e-12:
            return error_subarbol
        nodo.hijos.clear()
        nodo.proporciones.clear()
        nodo.atributo = nodo.tipo = nodo.umbral = None
        nodo.ganancia = nodo.split_info = nodo.gain_ratio = None
        nodo.motivo_parada = "pospoda"
        return error_hoja

    def _construir(self, caracteristicas, etiquetas, atributos, pesos, profundidad):
        """Construye y devuelve el Nodo raíz de un subárbol, con sus hijos si corresponde.

        Entradas internas ya preparadas por el entrenamiento:
            caracteristicas: filas que llegan a este nodo, con sus columnas originales.
            etiquetas y pesos: listas alineadas con esas filas por posición.
            atributos: columnas que todavía pueden utilizarse en este camino.
            profundidad: nivel actual, con la raíz en cero.
        Se presupone masa positiva y se conserva el orden global de clases.

        Primero resume las masas y aplica las paradas por pureza, atributos, nivel
        o masa mínima. Compara particiones categóricas y numéricas mediante Gain
        Ratio ajustado y aplica los umbrales de ganancia y razón por separado.
        Si se detiene devuelve una hoja con motivo_parada y distribución de clases.

        Al dividir, cada conocido entra únicamente en su rama con el peso recibido.
        Cada desconocido participa en todos los hijos con peso original * proporción
        de la rama, sin cambiar su clase ni sus otros atributos. Se conservan los
        aportes positivos y el orden; los índices de pandas no deciden el reparto.
        La suma de pesos de los hijos conserva la masa del padre y la de cada clase,
        salvo el redondeo numérico de las operaciones.

        Retira del camino el atributo categórico elegido y conserva el numérico,
        permitiendo nuevos cortes. Cada llamada descendiente aumenta un nivel.
        No modifica las tablas originales ni aprende de casos de validación o prueba.
        """
        # Resumir las clases con el orden global para resolver sus empates.
        masas_clases = conteos_ponderados(etiquetas, pesos)
        conteos = {clase: masas_clases[clase] for clase in self.classes_ if clase in masas_clases}
        clase_mayoritaria = max(conteos, key=conteos.get)
        nodo = Nodo(clase_mayoritaria, conteos, len(etiquetas), entropia(etiquetas, pesos),
                    peso_total=fsum(conteos.values()))

        # Detenerse cuando ya no hace falta o no está permitido seguir dividiendo.
        if len(conteos) == 1:
            nodo.motivo_parada = "clase_pura"
        elif not atributos:
            nodo.motivo_parada = "sin_atributos"
        elif self.profundidad_maxima is not None and profundidad >= self.profundidad_maxima:
            nodo.motivo_parada = "profundidad_maxima"
        elif nodo.peso_total < self.minimo_muestras_division:
            nodo.motivo_parada = "masa_menor_que_min_samples_split"
        if nodo.motivo_parada is not None:
            return nodo

        # Comparar las mejores divisiones de todos los atributos disponibles.
        candidatas = []
        for atributo in atributos:
            valores = caracteristicas[atributo].tolist()
            candidata = (buscar_mejor_umbral(valores, etiquetas, pesos, self.filtro_ganancia)
                         if self.tipos_[atributo] == "numerico"
                         else particion_categorica(valores, etiquetas, pesos))
            if candidata is None or candidata["split_info"] == 0:
                continue
            candidatas.append(dict(candidata, atributo=atributo))

        # El filtro opcional exige una ganancia al menos igual al promedio.
        # Se compara con una tolerancia pequeña para absorber redondeo numérico.
        if self.filtro_ganancia and candidatas:
            ganancia_media = fsum(candidata["ganancia"] for candidata in candidatas) / len(candidatas)
            candidatas = [candidata for candidata in candidatas
                          if candidata["ganancia"] >= ganancia_media - 1e-12]
        mejor_particion = None
        for candidata in candidatas:
            if mejor_particion is None or candidata["gain_ratio"] > mejor_particion["gain_ratio"]:
                mejor_particion = candidata
        if mejor_particion is None:
            nodo.motivo_parada = "sin_particion_valida"
            return nodo

        # Ganancia en bits y razón de ganancia tienen controles independientes.
        if mejor_particion["ganancia"] < self.umbral_ganancia:
            nodo.motivo_parada = "ganancia_menor_que_threshold"
            return nodo
        if mejor_particion["gain_ratio"] < self.minima_razon_ganancia:
            nodo.motivo_parada = "gain_ratio_menor_que_min_gain_ratio"
            return nodo

        # Guardar la decisión elegida antes de construir sus ramas.
        atributo = mejor_particion["atributo"]
        nodo.atributo = atributo
        nodo.tipo = self.tipos_[atributo]
        nodo.umbral = mejor_particion.get("umbral")
        nodo.ganancia = mejor_particion["ganancia"]
        nodo.split_info = mejor_particion["split_info"]
        nodo.gain_ratio = mejor_particion["gain_ratio"]
        nodo.proporciones = mejor_particion["proporciones"]
        atributos_siguientes = ([nombre for nombre in atributos if nombre != atributo]
                                if nodo.tipo == "categorico" else atributos)

        # Cada hijo recibe sus conocidos y la fracción correspondiente de los ausentes.
        posiciones_faltantes = set(mejor_particion["faltantes"])
        for rama, posiciones_conocidas in mejor_particion["ramas"].items():
            posiciones_conocidas = set(posiciones_conocidas)
            posiciones_hijo, pesos_hijo = [], []
            for posicion, peso in enumerate(pesos):
                nuevo_peso = peso if posicion in posiciones_conocidas else (
                    peso * nodo.proporciones[rama] if posicion in posiciones_faltantes else 0.0)
                if nuevo_peso > 0:
                    posiciones_hijo.append(posicion)
                    pesos_hijo.append(nuevo_peso)
            nodo.hijos[rama] = self._construir(
                caracteristicas.iloc[posiciones_hijo],
                [etiquetas[posicion] for posicion in posiciones_hijo],
                atributos_siguientes, pesos_hijo, profundidad + 1)
        return nodo

    def _distribucion_fila(self, fila, nodo):
        """Devuelve una lista de probabilidades de clase para una fila desde un nodo.

        Recibe fila, un diccionario atributo -> valor ya validado, y nodo, un Nodo
        aprendido con masa positiva. La lista devuelta sigue el orden global de
        classes_, incluyendo cero para clases sin masa en la hoja o nodo consultado.

        Una hoja devuelve sus frecuencias ponderadas. Un valor conocido sigue una
        rama: <= incluye el umbral y > los valores superiores. Si falta el atributo
        consultado, recorre todos los hijos y suma sus distribuciones multiplicadas
        por las proporciones aprendidas. No elige la rama más frecuente ni vota solo
        las clases ganadoras. Un faltante de otro atributo no afecta esta decisión.

        Una categoría conocida en formato pero nunca observada en ese nodo devuelve
        las frecuencias del nodo alcanzado, en lugar de mezclar sus hijos. Si todos
        los atributos están ausentes, la mezcla recupera la distribución del nodo
        desde el cual se consulta, salvo redondeo. Esta operación no reentrena ni
        normaliza al final: predecir_probabilidades realiza esa normalización.
        """
        # Una hoja responde con las proporciones de clases que aprendió.
        if nodo.es_hoja:
            return [nodo.conteos.get(clase, 0.0) / nodo.peso_total for clase in self.classes_]
        valor = fila[nodo.atributo]

        # Un ausente combina todas las distribuciones, no solo las clases ganadoras.
        if es_faltante(valor):
            distribuciones_hijas = [
                (nodo.proporciones[rama], self._distribucion_fila(fila, hijo))
                for rama, hijo in nodo.hijos.items()]
            return [fsum(proporcion * distribucion[posicion]
                         for proporcion, distribucion in distribuciones_hijas)
                    for posicion in range(len(self.classes_))]

        # Una categoría nueva usa el resumen local; un conocido continúa por su rama.
        rama = ("<=" if valor <= nodo.umbral else ">") if nodo.tipo == "numerico" else valor
        if rama not in nodo.hijos:
            return [nodo.conteos.get(clase, 0.0) / nodo.peso_total for clase in self.classes_]
        return self._distribucion_fila(fila, nodo.hijos[rama])

    def _validar_prediccion(self, caracteristicas):
        """Comprueba que la consulta conserve las columnas y sus tipos aprendidos."""
        # Predecir requiere un árbol ya entrenado y la misma estructura de entrada.
        if self.raiz_ is None:
            raise RuntimeError("El árbol debe entrenarse antes de predecir.")
        self._validar_caracteristicas(caracteristicas)
        if not caracteristicas.columns.equals(self.columnas_):
            raise ValueError("La consulta debe conservar las columnas y el orden de entrenamiento.")
        for atributo, tipo in self.tipos_.items():
            if tipo == "numerico" and any(
                    not es_faltante(valor)
                    and (not isinstance(valor, Real) or isinstance(valor, bool))
                    for valor in caracteristicas[atributo]):
                raise ValueError(f"El atributo {atributo!r} debe conservar su tipo numérico.")

    def predecir_probabilidades(self, caracteristicas):
        """Devuelve un DataFrame de probabilidades para los casos consultados.

        Entrada: caracteristicas con las mismas columnas, orden y significado
        numérico del entrenamiento; admite valores ausentes. Requiere un árbol
        entrenado. Devuelve una fila por caso y una columna por clase en classes_,
        conservando exactamente índices, duplicados y orden, sin modificar la entrada.

        Combina las rutas necesarias y divide cada distribución por su suma para
        absorber el redondeo acumulado. Las filas suman uno y expresan frecuencias
        ponderadas, no probabilidades calibradas. Se leen tuplas para conservar el
        tipo individual de los valores. La validación puede lanzar RuntimeError
        si falta entrenamiento, TypeError por formato o ValueError por datos inválidos.
        """
        # Leer tuplas conserva el tipo de cada valor, incluidos enteros grandes.
        self._validar_prediccion(caracteristicas)
        distribuciones = [
            self._distribucion_fila(dict(zip(self.columnas_, valores)), self.raiz_)
            for valores in caracteristicas.itertuples(index=False, name=None)]

        # Corregir únicamente el redondeo acumulado de las sumas fraccionarias.
        distribuciones = [[probabilidad / fsum(fila) for probabilidad in fila]
                          for fila in distribuciones]
        return pd.DataFrame(distribuciones, columns=self.classes_.copy(),
                            index=caracteristicas.index.copy())

    def predecir(self, caracteristicas):
        """Devuelve un DataFrame con una única columna prediccion.

        Recibe la misma tabla y aplica las mismas validaciones que
        predecir_probabilidades. Elige la clase de mayor probabilidad por caso;
        un empate conserva la primera clase en classes_, es decir, la primera
        observada en el entrenamiento completo. Conserva índice y orden sin
        modificar la consulta ni reentrenar. Los nombres predict y predict_proba
        acceden a estas mismas operaciones.
        """
        # Elegir la clase de mayor probabilidad; el orden global resuelve empates.
        probabilidades = self.predecir_probabilidades(caracteristicas)
        clases = [self.classes_[max(range(len(fila)), key=fila.__getitem__)]
                  for fila in probabilidades.itertuples(index=False, name=None)]
        return pd.DataFrame({"prediccion": clases}, index=caracteristicas.index.copy())

    # Accesos compatibles al mismo código para entrenamiento e inferencia.
    fit = entrenar
    predict = predecir
    predict_proba = predecir_probabilidades
